In [ ]:
import logging
import os
from pathlib import Path
from pydantic import BaseModel, Field, model_validator
from typing import Optional, override

from sqlite_manager.crud import CRUDBase
from sqlite_manager.interface import SQLiteInterface

%load_ext autoreload
%autoreload 2

PROJECT_DIR = Path(os.getcwd()).resolve()
DB_PATH = PROJECT_DIR / "products.sqlite3"

logging.basicConfig(level=logging.INFO)


def create_product_database() -> SQLiteInterface:
    """Creates the product database and creates the products table."""

    sql_db = SQLiteInterface(DB_PATH)
    sql_db.execute_sql(
        """
      CREATE TABLE IF NOT EXISTS products (
        product_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        price REAL NOT NULL,
        description TEXT,
        price_with_tax REAL
      )
      """
    )
    logging.info(f"Database initialized at {DB_PATH}")

    return sql_db


# Define Pydantic model
class Product(BaseModel):
    product_id: int = Field(default=None)
    name: str
    price: float
    description: Optional[str] = None
    price_with_tax: Optional[float] = None

    @model_validator(mode="after")
    def validate_price_with_tax(self) -> "Product":
        """Validate the price with tax calculation"""
        if self.price is not None:
            self.price_with_tax = self.price * 1.2
        return self


# Create a CRUD handler with Pydantic model support
class PydanticCRUD(CRUDBase[Product]):  # Generic type support
    """CRUD handler that returns Pydantic models"""

    def __init__(self, sql_db, table_name="products", id_column="product_id"):
        super().__init__(sql_db, table_name, id_column)

    @override
    def row_factory(self, cursor, row) -> Product:
        """Returns a Pydantic model instance from a row"""
        fields = [column[0] for column in cursor.description]

        return Product(**dict(zip(fields, row, strict=True)))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
# Create example datbase
sql_db = create_product_database()

# Create the CRUD handler
crud = PydanticCRUD(sql_db)

INFO:root:Database initialized at /home/jverster/Projects/Other/sqlite-manager/examples/pydantic_validation/products.sqlite


In [22]:
# Create product
crud.create(
    name="Example Product", price=100.0, description="This is an example product"
)

INFO:sqlite_manager.crud:Inserted new record into products


True

In [ ]:
product = Product(
    name="Example Product 2",
    price=200.0,
    description="This is another example product",
)
crud.create(**product.model_dump(exclude_unset=True))

INFO:sqlite_manager.crud:Inserted new record into products


True

In [27]:
crud.read({"product_id": 1})  # Returns a Product instance

Product(product_id=1, name='Example Product', price=100.0, description='This is an example product', price_with_tax=120.0)

In [28]:
crud.read({"product_id": 2}).model_dump()  # Returns a dict

{'product_id': 2,
 'name': 'Example Product',
 'price': 100.0,
 'description': 'This is an example product',
 'price_with_tax': 120.0}